# Review validation and test results
PRAUC in older runs means average precision (AP). New runs use AP in the logs. The validation heads below are diagnostics; the three test conditions all use the **same final prediction head and checkpoint**. NaN scores are excluded from macro averages. Positive-only findings can have AP=1 without meaningful negative discrimination.

In [16]:
from pathlib import Path
import sys

import pandas as pd
import torch
from IPython.display import display

# Works when launched from Research, Pipeline2, or model_notebooks.
PROJECT_DIR = next(
    folder if (folder / "main.py").is_file() else folder / "Pipeline2"
    for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "models" / "main_trainer.py").is_file()
    or (folder / "Pipeline2" / "models" / "main_trainer.py").is_file()
)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from utils.data_helper import TARGETS

# Change this to review a different run.
RUN_DIR = PROJECT_DIR / "runs_complete" / "version_3"
# Test result metadata points to the actual selected checkpoint (also after resume).
result_path = RUN_DIR / "test_results_all.pt"
if result_path.exists():
    payload = torch.load(result_path, map_location="cpu", weights_only=False)
    CHECKPOINT_PATH = Path(payload["best_checkpoint"])
    del payload
else:
    # Lightning may version the filename after replacing an earlier best file.
    candidates = list((RUN_DIR / "checkpoints").glob("best*.ckpt"))
    CHECKPOINT_PATH = max(candidates, key=lambda path: path.stat().st_mtime) if candidates else RUN_DIR / "checkpoints/best.ckpt"
metrics = pd.read_csv(RUN_DIR / "metrics.csv")

# Load only your own trusted checkpoint; keep it on CPU.
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False, mmap=True)
best_epoch = int(checkpoint["epoch"])
best_step = int(checkpoint["global_step"])
del checkpoint  # Release the model and optimizer tensors from memory.
print("Run:", RUN_DIR)
print("Best checkpoint:", CHECKPOINT_PATH)
print("Checkpoint epoch index:", best_epoch)
print("Completed epochs at checkpoint:", best_epoch + 1)
print("Optimizer steps at checkpoint:", best_step)

Run: /home/rittick-dutta/Desktop/Research/Pipeline2/runs_complete/version_3
Best checkpoint: /home/rittick-dutta/Desktop/Research/Pipeline2/runs_complete/version_3/checkpoints/best.ckpt
Checkpoint epoch index: 15
Completed epochs at checkpoint: 16
Optimizer steps at checkpoint: 20000


## Validation scores after each completed epoch
Lightning writes training and validation values to separate CSV rows. Select the rows containing final-model validation scores.

In [17]:
AP_PREFIX = "val_AP" if "val_AP_avg_over_dxs/final" in metrics else "val_PRAUC"
ap_column = f"{AP_PREFIX}_avg_over_dxs/final"
roc_column = "val_AUROC_avg_over_dxs/final"
validation = pd.DataFrame()
if ap_column in metrics.columns:
    validation = (
        metrics.dropna(subset=[ap_column])
        .sort_values(["epoch", "step"])
        .drop_duplicates(subset="epoch", keep="last")
        .copy()
    )
    validation["epoch"] = validation["epoch"].astype(int)
    
    history = validation[["epoch", ap_column, roc_column]].rename(columns={
        "epoch": "epoch_index",
        ap_column: "validation_macro_AP",
        roc_column: "validation_macro_AUROC",
    })
    history.insert(1, "completed_epochs", history["epoch_index"] + 1)
    history["saved_best_checkpoint"] = history["epoch_index"].eq(best_epoch)
    display(history.reset_index(drop=True).round(4))
else:
    print("This version contains test evaluations only; validation logs are in the training run.")

# Read the metrics belonging to the checkpoint, rather than the latest epoch.
matching = validation.loc[validation["epoch"].eq(best_epoch)] if not validation.empty else validation
best_row = matching.iloc[-1] if len(matching) else None
if best_row is not None:
    print(f"Best checkpoint: AP={best_row[ap_column]:.4f}, AUROC={best_row[roc_column]:.4f}")
else:
    print("This checkpoint was selected in an earlier training version. Review that version's validation logs for its scores.")

,epoch_index,completed_epochs,validation_macro_AP,validation_macro_AUROC,saved_best_checkpoint
0,14,15,0.8523,0.7332,False
1,15,16,0.8614,0.7309,True
2,16,17,0.8580,0.7409,False


Best checkpoint: AP=0.8614, AUROC=0.7309


## Best checkpoint validation heads
New runs validate with all inputs available. The final head is the combined output; ECG, labs and CXR heads provide auxiliary diagnostics. These rows are different from the three final-head test conditions.

In [18]:
if best_row is not None:
    head_scores = pd.DataFrame([
        {
            "prediction_head": head,
            "validation_macro_AP": best_row[f"{AP_PREFIX}_avg_over_dxs/{head}"],
            "validation_macro_AUROC": best_row[f"val_AUROC_avg_over_dxs/{head}"],
        }
        for head in ["final", "ecg", "labs", "cxr"]
    ])
    display(head_scores.round(4))


,prediction_head,validation_macro_AP,validation_macro_AUROC
0,final,0.8614,0.7309
1,ecg,0.7970,0.6051
2,labs,0.8226,0.6145
3,cxr,0.8549,0.7464


## Best checkpoint: all 14 findings
These are the final fused model’s per-finding validation scores. No Finding can have AP=1 while AUROC is undefined when its observed labels contain only positives.

In [19]:
if best_row is not None:
    finding_scores = pd.DataFrame({
        "finding": TARGETS,
        "validation_AP": [best_row[f"{AP_PREFIX}_per_dx_final/{name}"] for name in TARGETS],
        "validation_AUROC": [best_row[f"val_AUROC_per_dx_final/{name}"] for name in TARGETS],
    })
    display(finding_scores.round(4))
    print("Findings included in macro AP:", finding_scores["validation_AP"].notna().sum())
    print("Findings included in macro AUROC:", finding_scores["validation_AUROC"].notna().sum())


,finding,validation_AP,validation_AUROC
0,Atelectasis,0.9983,0.9289
1,Cardiomegaly,0.9089,0.7893
2,Consolidation,0.8642,0.7091
3,Edema,0.7618,0.7947
4,Enlarged Cardiomediastinum,0.7721,0.6776
5,Fracture,0.9651,0.5714
6,Lung Lesion,1.0000,NaN
7,Lung Opacity,0.9832,0.7832
8,No Finding,1.0000,NaN
9,Pleural Effusion,0.8777,0.8224


Findings included in macro AP: 14
Findings included in macro AUROC: 11


## Final head under three test conditions
Generated after testing completes. Every row uses the same checkpoint, admission IDs and labels.

In [20]:
for filename in ["test_summary.csv", "test_differences.csv", "test_per_finding.csv"]:
    path = RUN_DIR / filename
    if path.exists():
        print(filename)
        display(pd.read_csv(path).round(4))
    else:
        print(filename, "will be available after all three test evaluations finish.")

test_summary.csv


,condition,modalities,admissions,macro_AP,macro_AUROC,AP_findings,AUROC_findings,delta_AP_vs_all,delta_AUROC_vs_all
0,all,cxr+ecg+labs,464,0.8631,0.7477,14,11,0.0000,0.0000
1,cxr_only,cxr,464,0.8609,0.7278,14,11,-0.0022,-0.0199
2,ecg_labs,ecg+labs,464,0.8115,0.6343,14,11,-0.0515,-0.1134


test_differences.csv


,comparison,delta_AP,delta_AUROC
0,cxr_only minus all,-0.0022,-0.0199
1,ecg_labs minus all,-0.0515,-0.1134
2,ecg_labs minus cxr_only,-0.0494,-0.0935


test_per_finding.csv


,finding,positive,negative,uncertain,unreported,AP_all,AUROC_all,AP_cxr_only,AUROC_cxr_only,AP_ecg_labs,AUROC_ecg_labs
0,Atelectasis,141,5,22,296,0.9834,0.6908,0.9795,0.6482,0.9916,0.7872
1,Cardiomegaly,144,50,20,250,0.9145,0.7981,0.9167,0.7919,0.8101,0.6264
2,Consolidation,29,22,8,405,0.8224,0.7931,0.8025,0.7900,0.8030,0.7053
3,Edema,82,88,39,255,0.8353,0.8444,0.8258,0.8307,0.6585,0.6912
4,Enlarged Cardiomediastinum,28,15,30,391,0.8109,0.6690,0.8166,0.6619,0.7211,0.5643
5,Fracture,10,0,0,454,1.0000,NaN,1.0000,NaN,1.0000,NaN
6,Lung Lesion,12,2,3,447,1.0000,1.0000,1.0000,1.0000,0.9253,0.6667
7,Lung Opacity,131,13,12,308,0.9722,0.7933,0.9701,0.7827,0.9275,0.5291
8,No Finding,69,0,0,395,1.0000,NaN,1.0000,NaN,1.0000,NaN
9,Pleural Effusion,166,90,15,193,0.8879,0.8131,0.8882,0.8177,0.7773,0.6722


## Original label counts
Counts distinguish explicit negatives from unknown labels. They are saved before training begins and cover train, validation and clinical test. Evaluation-only runs report test counts.

In [21]:
path = RUN_DIR / "label_counts.csv"
if path.exists():
    counts = pd.read_csv(path)
    for split, table in counts.groupby("split", sort=False):
        print(split)
        display(table.reset_index(drop=True))
else:
    print("No label-count file in this older run.")

train


,finding,positive,negative,uncertain,unreported,split
0,Atelectasis,2867,118,588,6427,train
1,Cardiomegaly,3346,886,450,5318,train
2,Consolidation,741,330,263,8666,train
3,Edema,2067,1772,834,5327,train
4,Enlarged Cardiomediastinum,504,265,663,8568,train
5,Fracture,179,17,12,9792,train
6,Lung Lesion,285,22,52,9641,train
7,Lung Opacity,3108,175,223,6494,train
8,No Finding,1368,0,0,8632,train
9,Pleural Effusion,3618,1701,371,4310,train


val


,finding,positive,negative,uncertain,unreported,split
0,Atelectasis,225,5,43,477,val
1,Cardiomegaly,268,81,32,369,val
2,Consolidation,60,22,21,647,val
3,Edema,148,151,52,399,val
4,Enlarged Cardiomediastinum,38,20,50,642,val
5,Fracture,14,1,0,735,val
6,Lung Lesion,18,0,2,730,val
7,Lung Opacity,220,13,12,505,val
8,No Finding,113,0,0,637,val
9,Pleural Effusion,253,149,27,321,val


test


,finding,positive,negative,uncertain,unreported,split
0,Atelectasis,141,5,22,296,test
1,Cardiomegaly,144,50,20,250,test
2,Consolidation,29,22,8,405,test
3,Edema,82,88,39,255,test
4,Enlarged Cardiomediastinum,28,15,30,391,test
5,Fracture,10,0,0,454,test
6,Lung Lesion,12,2,3,447,test
7,Lung Opacity,131,13,12,308,test
8,No Finding,69,0,0,395,test
9,Pleural Effusion,166,90,15,193,test
